<a href="https://colab.research.google.com/github/istanbuldatascienceacademy/cmeep2026/blob/main/02_demo_sorgulari_surum.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PISA 2025 · Demo sorguları — SÜRÜM 3 (9 Ekim, 12:55)
**CMEEP 2026 · Çalıştay 14 · Zekeriya Beşiroğlu · AI Architect · İstanbul Data Science Academy**

Kitteki tüm SQL'ler, sunum sırasıyla, her biri ayrı hücrede. **Kopyala–yapıştır yok:** hücredeki ▶ tuşuna basın, sonuç tablosu ve **taranan veri** altında görünür.

1. **1. hücreye** proje kimliğinizi yazın ve ilk iki hücreyi çalıştırın.
2. **Bölüm 0'ı DEMODAN ÖNCE bir kez çalıştırın:** modeller orada eğitilir (~10–15 dk).
3. Demoda **Bölüm 1'den itibaren** sırayla ilerleyin. Başlıklardaki numara, sunumdaki slayt numarasıdır.

Önkoşul: `01_pisa_bigquery_yukle.ipynb` defteri çalıştırılmış olmalı (`pisa.ogrenci`, `pisa.okul`).

In [ ]:
PROJE_ID = "atolye-0926"   # ← 01 defterindekiyle aynı

In [ ]:
from google.colab import auth
auth.authenticate_user()
from google.cloud import bigquery
import pandas as pd
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 250)
import re
client = bigquery.Client(project=PROJE_ID)
VERI_SETI = "pisa"


def tam_ad(sql):
    """`pisa.tablo` → `proje.pisa.tablo` (CREATE MODEL gibi komutlar proje adını ister)."""
    return re.sub(r"`pisa\.", f"`{PROJE_ID}.{VERI_SETI}.", sql)


def calistir(sql, onbellek=True):
    """Sorguyu çalıştırır; sonucu tablo olarak gösterir, taranan veri ve süreyi yazar."""
    if sql is None:
        print("↷ Atlandı: uygun karşılaştırma ülkesi yok (Özellik kontrolü çıktısına bakın).")
        return None
    ayar = bigquery.QueryJobConfig(use_query_cache=onbellek,
                                   default_dataset=f"{PROJE_ID}.{VERI_SETI}")
    is_ = client.query(tam_ad(doldur(sql)), job_config=ayar)
    df = is_.to_dataframe()
    mb = (is_.total_bytes_processed or 0) / 1e6
    sure = (is_.ended - is_.started).total_seconds() if is_.ended and is_.started else 0
    print(f"✔ {mb:,.1f} MB tarandı · {sure:.1f} sn · önbellek: {'evet' if is_.cache_hit else 'hayır'}")
    return df


def tahmini_tarama(sql):
    """Sorguyu ÇALIŞTIRMADAN kaç MB tarayacağını gösterir (maliyet demosu için)."""
    is_ = client.query(tam_ad(doldur(sql)), job_config=bigquery.QueryJobConfig(
        dry_run=True, use_query_cache=False, default_dataset=f"{PROJE_ID}.{VERI_SETI}"))
    print(f"Bu sorgu {is_.total_bytes_processed / 1e6:,.1f} MB tarayacak "
          f"(~{is_.total_bytes_processed / 2**40 * 6.25:.4f} $; ilk 1 TiB/ay ücretsiz).")


print("Hazır. Proje:", PROJE_ID)

Hazır. Proje: atolye-0926


### Özellik kontrolü (önemli)
PISA 2025'in ağırlıklı alanı **fen** olduğundan matematiğe özgü endeksler (`ANXMAT`, `MATHEFF`) 2025 dosyasında bulunmayabilir.
Bu hücre `pisa.ogrenci` tablosundaki sütunlara bakar ve **modellerde yalnızca var olan değişkenleri** kullanır.
2025'e özgü bir endeksi (ör. fen öz-yeterliği) eklemek isterseniz önce 01 defterinde `ANKET` listesine ekleyip yeniden yükleyin,
sonra adını aşağıdaki `EK_OZELLIKLER` listesine yazın.

In [ ]:
EK_OZELLIKLER = []                        # ← 2025'e özgü ek endeks adları (varsa)
KARSILASTIRMA_ADAY = ["GRC", "ITA", "ESP", "PRT", "POL"]   # Türkiye modelinin sınanacağı ülkeler
ESIK = 0.5                                     # bir değişken ülkede en az %50 dolu olmalı

sutunlar = {r.column_name for r in client.query(
    f"SELECT column_name FROM `{PROJE_ID}.{VERI_SETI}.INFORMATION_SCHEMA.COLUMNS` "
    f"WHERE table_name = 'ogrenci'").result()}

ISTENEN_OZ = ["ESCS", "HOMEPOS", "BELONG", "ANXMAT", "MATHEFF", "REPEAT"] + EK_OZELLIKLER
YEDEK_OZ = ["ICTRES", "DISCLIM", "TEACHSUP", "FAMSUP", "EXPOFA"]
adaylar = [d for d in ISTENEN_OZ + YEDEK_OZ + ["ST004D01T"] if d in sutunlar]

# Her değişkenin her ülkedeki doluluk oranı (yalnızca dolu sütunlarla model kurulur)
ulke_listesi = ", ".join(f"'{u}'" for u in ["TUR"] + KARSILASTIRMA_ADAY)
kapsama = client.query(
    "SELECT CNT, COUNT(*) AS ogrenci, "
    + ", ".join(f"COUNTIF({d} IS NOT NULL) / COUNT(*) AS {d}" for d in adaylar)
    + f" FROM `{PROJE_ID}.{VERI_SETI}.ogrenci` WHERE CYCLE = 2025 AND CNT IN ({ulke_listesi})"
    + " GROUP BY CNT").to_dataframe().set_index("CNT")
print("Doluluk oranları (1 = tamamen dolu, 0 = tamamen boş):")
display(kapsama.round(2))
if "TUR" not in kapsama.index:
    raise ValueError("Tabloda 2025 Türkiye verisi yok: 01 defterini kontrol edin.")


def dolu(d, ulke):
    return d in kapsama.columns and ulke in kapsama.index and kapsama.loc[ulke, d] >= ESIK


# Model özellikleri: Türkiye'de dolu olanlar (en az 4; gerekirse yedeklerden)
OZ = [d for d in ISTENEN_OZ if dolu(d, "TUR")]
if len(OZ) < 4:
    OZ += [d for d in YEDEK_OZ if dolu(d, "TUR") and d not in OZ][: 4 - len(OZ)]
CINSIYET = dolu("ST004D01T", "TUR")
PROFIL = [d for d in ["ESCS", "BELONG", "ANXMAT", "MATHEFF"] + EK_OZELLIKLER if dolu(d, "TUR")]
if len(PROFIL) < 3:
    PROFIL += [d for d in ["HOMEPOS"] + YEDEK_OZ if dolu(d, "TUR") and d not in PROFIL][: 3 - len(PROFIL)]

# Karşılaştırma ülkeleri: 2025'e katılmış VE tüm model özellikleri dolu olanlar
gerekli = OZ + (["ST004D01T"] if CINSIYET else [])
KARSI = [u for u in KARSILASTIRMA_ADAY if all(dolu(d, u) for d in gerekli)][:3]

print("Model özellikleri  :", ", ".join(OZ) + (", cinsiyet" if CINSIYET else ""))
print("Profil özellikleri :", ", ".join(PROFIL))
print("Karşılaştırma ülkeleri:", ", ".join(KARSI) if KARSI else "yok (slayt 31 atlanır)")
cikarilan = [d for d in ISTENEN_OZ if d not in OZ]
if cikarilan:
    print("Türkiye 2025'te bulunmadığı ya da boş olduğu için çıkarılanlar:", ", ".join(cikarilan))


def karsi_model_sql():
    """Aynı risk modelini ilk karşılaştırma ülkesinde eğitir (slayt 31)."""
    if not KARSI:
        return None
    return f"""
CREATE OR REPLACE MODEL `pisa.m_risk_karsi`
OPTIONS (
  model_type = 'BOOSTED_TREE_CLASSIFIER',
  input_label_cols = ['duzey2_alti'],
  data_split_method = 'CUSTOM',
  data_split_col = 'test_mi',
  max_iterations = 50,
  enable_global_explain = TRUE
) AS
SELECT
  IF(PV1MATH < 420.07, 1, 0) AS duzey2_alti,
  {{OZELLIKLER}},
  MOD(ABS(FARM_FINGERPRINT(CAST(CNTSTUID AS STRING))), 5) = 0 AS test_mi
FROM `pisa.ogrenci`
WHERE CNT = '{KARSI[0]}' AND CYCLE = 2025"""


def karsi_sinama_sql():
    """Türkiye modelini karşılaştırma ülkelerinde sınar (slayt 31)."""
    if not KARSI:
        return None
    parcalar = ["SELECT 'TUR modeli → TUR (test kümesi)' AS sinama, precision, recall, f1_score, roc_auc\n"
                "FROM ML.EVALUATE(MODEL `pisa.m_risk_2025`)"]
    for u in KARSI:
        parcalar.append(f"""SELECT 'TUR modeli → {u}', precision, recall, f1_score, roc_auc
FROM ML.EVALUATE(MODEL `pisa.m_risk_2025`,
  (SELECT IF(PV1MATH < 420.07, 1, 0) AS duzey2_alti, {{OZELLIKLER}}
   FROM `pisa.ogrenci` WHERE CNT = '{u}' AND CYCLE = 2025))""")
    return "\nUNION ALL\n".join(parcalar)


def karsi_onem_sql():
    """Türkiye ve karşılaştırma ülkesi modellerinde değişken önemi yan yana (slayt 31)."""
    if not KARSI:
        return None
    u = KARSI[0]
    return f"""
SELECT feature,
  ROUND(MAX(IF(model = 'TUR', attribution, NULL)), 3) AS onem_TUR,
  ROUND(MAX(IF(model = '{u}', attribution, NULL)), 3) AS onem_{u}
FROM (
  SELECT 'TUR' AS model, feature, attribution FROM ML.GLOBAL_EXPLAIN(MODEL `pisa.m_risk_2025`)
  UNION ALL
  SELECT '{u}', feature, attribution FROM ML.GLOBAL_EXPLAIN(MODEL `pisa.m_risk_karsi`)
)
GROUP BY feature
ORDER BY onem_TUR DESC"""


def doldur(sql):
    """Sorgulardaki {OZELLIKLER}, {PROFIL}, {PROFIL_IN} jetonlarını mevcut sütunlarla doldurur."""
    oz = ", ".join(OZ) + (",\n    CAST(CAST(ST004D01T AS INT64) AS STRING) AS cinsiyet" if CINSIYET else "")
    return (sql.replace("{OZELLIKLER}", oz)
               .replace("{PROFIL_IN}", ", ".join(f"'{p}'" for p in PROFIL))
               .replace("{PROFIL}", ", ".join(PROFIL)))


def senaryo_sql():
    """Yalnızca mevcut değişkenlerle 'ne olurdu?' senaryoları kurar (slayt 33)."""
    kolonlar = OZ + (["cinsiyet"] if CINSIYET else [])
    senaryolar = [("0 Mevcut durum", {})]
    if "BELONG" in OZ:
        senaryolar.append(("1 Okul aidiyeti +0,5 SS", {"BELONG": "BELONG + 0.5"}))
    if "ANXMAT" in OZ:
        senaryolar.append(("2 Matematik kaygısı −0,5 SS", {"ANXMAT": "ANXMAT - 0.5"}))
    if "MATHEFF" in OZ:
        senaryolar.append(("3 Matematik öz-yeterliği +0,5 SS", {"MATHEFF": "MATHEFF + 0.5"}))
    for ek in EK_OZELLIKLER:
        if ek in OZ:
            senaryolar.append((f"E {ek} +0,5 SS", {ek: f"{ek} + 0.5"}))
    if "REPEAT" in OZ:
        senaryolar.append(("4 Sınıf tekrarı olmasaydı", {"REPEAT": "0.0"}))
    if "ESCS" in OZ:
        destek = {"ESCS": "IF(ESCS < -1, ESCS + 0.5, ESCS)"}
        if "HOMEPOS" in OZ:
            destek["HOMEPOS"] = "IF(ESCS < -1, HOMEPOS + 0.5, HOMEPOS)"
        senaryolar.append(("5 Dezavantajlılara destek (ESCS < −1 için +0,5)", destek))
    paket = {}
    for _, d in senaryolar[1:]:
        if any(k in d for k in ("BELONG", "ANXMAT", "MATHEFF")):
            paket.update(d)
    if len(paket) > 1:
        senaryolar.append(("6 Birleşik paket (aidiyet + kaygı + öz-yeterlik)", paket))
    satirlar = []
    for ad, degisim in senaryolar:
        ifadeler = ", ".join(f"{degisim.get(k, k)} AS {k}" for k in kolonlar)
        satirlar.append(f"  SELECT '{ad}' AS senaryo, W_FSTUWT, {ifadeler} FROM taban")
    return f"""
WITH taban AS (
  SELECT W_FSTUWT, {{OZELLIKLER}}
  FROM `pisa.ogrenci`
  WHERE CNT = 'TUR' AND CYCLE = 2025
),
senaryolar AS (
{chr(10).join(['  UNION ALL' + chr(10) + s if i else s for i, s in enumerate(satirlar)])}
),
tahmin AS (
  SELECT senaryo, W_FSTUWT,
    (SELECT prob FROM UNNEST(predicted_duzey2_alti_probs) WHERE label = 1) AS risk
  FROM ML.PREDICT(MODEL `pisa.m_risk_2025`, (SELECT * FROM senaryolar))
),
ozet AS (
  SELECT senaryo, 100 * SUM(W_FSTUWT * risk) / SUM(W_FSTUWT) AS risk_yuzde
  FROM tahmin GROUP BY senaryo
)
SELECT senaryo,
  ROUND(risk_yuzde, 1) AS duzey2_alti_tahmini_yuzde,
  ROUND(risk_yuzde - MAX(IF(senaryo = '0 Mevcut durum', risk_yuzde, NULL)) OVER (), 1) AS mevcuda_gore_puan
FROM ozet
ORDER BY senaryo"""


Doluluk oranları (1 = tamamen dolu, 0 = tamamen boş):


,ogrenci,ESCS,HOMEPOS,BELONG,REPEAT,ICTRES,TEACHSUP,FAMSUP,ST004D01T
CNT,,,,,,,,,
ITA,8946,0.97,0.99,0.99,0.99,0.00,0.98,0.95,1.0
PRT,6945,0.95,0.98,0.97,0.97,0.00,0.93,0.91,1.0
POL,7103,0.95,0.98,0.97,0.98,0.00,0.95,0.91,1.0
ESP,29966,0.95,0.97,0.95,0.96,0.97,0.90,0.89,0.0
GRC,6858,0.96,0.99,0.97,0.98,0.00,0.97,0.92,1.0
TUR,7702,0.97,1.00,0.99,0.99,1.00,0.99,0.98,1.0


Model özellikleri  : ESCS, HOMEPOS, BELONG, REPEAT, cinsiyet
Profil özellikleri : ESCS, BELONG, HOMEPOS
Karşılaştırma ülkeleri: GRC, ITA, PRT
Türkiye 2025'te bulunmadığı ya da boş olduğu için çıkarılanlar: ANXMAT, MATHEFF


---
## 0 · Hazırlık: modelleri eğit (DEMODAN ÖNCE, bir kez)
Her hücre birkaç dakika sürebilir. `CREATE MODEL` hata verirse projede faturalandırmayı açıp bütçe uyarısı koyun.
`Unrecognized name` hatası 2025 dosyasında o değişkenin olmadığını gösterir.

**⏳ 06_model_risk.sql · A) Yorumlanabilir taban model: lojistik regresyon  (~1 dk)** → `pisa.m_risk_lojistik`

In [ ]:
calistir(r"""
CREATE OR REPLACE MODEL `pisa.m_risk_lojistik`
OPTIONS (
  model_type = 'LOGISTIC_REG',
  input_label_cols = ['duzey2_alti'],
  data_split_method = 'CUSTOM',
  data_split_col = 'test_mi',
  enable_global_explain = TRUE
) AS
SELECT
  IF(PV1MATH < 420.07, 1, 0) AS duzey2_alti,
  {OZELLIKLER},
  MOD(ABS(FARM_FINGERPRINT(CAST(CNTSTUID AS STRING))), 5) = 0 AS test_mi
FROM `pisa.ogrenci`
WHERE CNT = 'TUR' AND CYCLE = 2025
""")

✔ 24.9 MB tarandı · 23.7 sn · önbellek: hayır


""


**⏳ 06_model_risk.sql · B) Gradyan artırmalı ağaç  (~3–5 dk)** → `pisa.m_risk_2025`

In [ ]:
calistir(r"""
CREATE OR REPLACE MODEL `pisa.m_risk_2025`
OPTIONS (
  model_type = 'BOOSTED_TREE_CLASSIFIER',
  input_label_cols = ['duzey2_alti'],
  data_split_method = 'CUSTOM',
  data_split_col = 'test_mi',
  max_iterations = 50,
  enable_global_explain = TRUE
) AS
SELECT
  IF(PV1MATH < 420.07, 1, 0) AS duzey2_alti,
  {OZELLIKLER},
  MOD(ABS(FARM_FINGERPRINT(CAST(CNTSTUID AS STRING))), 5) = 0 AS test_mi
FROM `pisa.ogrenci`
WHERE CNT = 'TUR' AND CYCLE = 2025
""")

✔ 14,281.5 MB tarandı · 273.8 sn · önbellek: hayır


""


**⏳ 07_ulkeler_arasi_aktarim.sql · Aynı modeli karşılaştırma ülkesinde eğit** → `pisa.m_risk_karsi`

In [ ]:
calistir(karsi_model_sql())

✔ 14,281.0 MB tarandı · 274.6 sn · önbellek: hayır


""


**⏳ 08_kmeans_profiller.sql · A) Model** → `pisa.m_profil_2025`

In [ ]:
calistir(r"""
CREATE OR REPLACE MODEL `pisa.m_profil_2025`
OPTIONS (
  model_type = 'KMEANS',
  num_clusters = 4,
  kmeans_init_method = 'KMEANS++',
  standardize_features = TRUE
) AS
SELECT {PROFIL}
FROM `pisa.ogrenci`
WHERE CNT = 'TUR' AND CYCLE = 2025
""")

✔ 12.8 MB tarandı · 45.2 sn · önbellek: hayır


""


---
## Slayt 18 · Yükleme kontrolü
`sql/01_kontrol.sql`

> 01 · Yükleme kontrolü
> Slayt: "PISA verisini BigQuery'ye taşımak"
> Ne gösterir: Her döngüde kaç öğrenci, kaç ülke var; Türkiye satırları.
> Maliyet: birkaç MB (yalnızca 3 sütun okunur)

**Maliyet demosu:** sorguyu çalıştırmadan önce kaç MB tarayacağını görün.

In [ ]:
tahmini_tarama(r"""
SELECT
  CYCLE                         AS dongu,
  COUNT(*)                      AS ogrenci_sayisi,
  COUNT(DISTINCT CNT)           AS ulke_sayisi,
  COUNTIF(CNT = 'TUR')          AS turkiye_ogrenci,
  ROUND(SUM(IF(CNT = 'TUR', W_FSTUWT, 0))) AS turkiye_temsil_edilen_15_yas
FROM `pisa.ogrenci`
GROUP BY dongu
ORDER BY dongu
""")

Bu sorgu 15.9 MB tarayacak (~0.0001 $; ilk 1 TiB/ay ücretsiz).


In [ ]:
calistir(r"""
SELECT
  CYCLE                         AS dongu,
  COUNT(*)                      AS ogrenci_sayisi,
  COUNT(DISTINCT CNT)           AS ulke_sayisi,
  COUNTIF(CNT = 'TUR')          AS turkiye_ogrenci,
  ROUND(SUM(IF(CNT = 'TUR', W_FSTUWT, 0))) AS turkiye_temsil_edilen_15_yas
FROM `pisa.ogrenci`
GROUP BY dongu
ORDER BY dongu
""")

✔ 0.0 MB tarandı · 0.1 sn · önbellek: evet


,dongu,ogrenci_sayisi,ulke_sayisi,turkiye_ogrenci,turkiye_temsil_edilen_15_yas
0,2025,755721,90,7702,920900.0


---
## Slayt 19 · Ülke ortalamaları (10 PV)
`sql/02_ulke_ortalamalari.sql`

> 02 · Ülke ortalamaları: 10 olası değer + tam örneklem ağırlığı
> Slayt: "Olası değerlerle ülke ortalaması"
> DOĞRULAMA (resmi raporlarla):
>   TUR 2025 → matematik 462 · okuma 472 · fen 494   (OECD ort. 463 · 461 · 482)
> Döngüyü değiştirmek için aşağıdaki "CYCLE = 2025" ifadesini düzenleyin.
> OECD ortalaması satırı dosyadaki OECD değişkeniyle hesaplanır; üye
> kümesi raporla aynı değilse birkaç puan sapabilir.

In [ ]:
calistir(r"""
WITH secim AS (            -- yalnızca gereken sütunlar: daha az bayt, daha az maliyet
  SELECT CYCLE, CNT, OECD, W_FSTUWT,
    PV1MATH, PV2MATH, PV3MATH, PV4MATH, PV5MATH,
    PV6MATH, PV7MATH, PV8MATH, PV9MATH, PV10MATH,
    PV1READ, PV2READ, PV3READ, PV4READ, PV5READ,
    PV6READ, PV7READ, PV8READ, PV9READ, PV10READ,
    PV1SCIE, PV2SCIE, PV3SCIE, PV4SCIE, PV5SCIE,
    PV6SCIE, PV7SCIE, PV8SCIE, PV9SCIE, PV10SCIE
  FROM `pisa.ogrenci`
  WHERE CYCLE = 2025
),
uzun AS (                  -- 10 olası değeri satırlara çevir
  SELECT * FROM secim
  UNPIVOT ((mat, oku, fen) FOR pv_no IN (
  (PV1MATH, PV1READ, PV1SCIE) AS 1,
  (PV2MATH, PV2READ, PV2SCIE) AS 2,
  (PV3MATH, PV3READ, PV3SCIE) AS 3,
  (PV4MATH, PV4READ, PV4SCIE) AS 4,
  (PV5MATH, PV5READ, PV5SCIE) AS 5,
  (PV6MATH, PV6READ, PV6SCIE) AS 6,
  (PV7MATH, PV7READ, PV7SCIE) AS 7,
  (PV8MATH, PV8READ, PV8SCIE) AS 8,
  (PV9MATH, PV9READ, PV9SCIE) AS 9,
  (PV10MATH, PV10READ, PV10SCIE) AS 10))
),
pv_ort AS (                -- her döngü × ülke × PV için ağırlıklı ortalama
  SELECT CYCLE, CNT, ANY_VALUE(OECD) AS oecd, pv_no,
    SUM(W_FSTUWT * mat) / SUM(W_FSTUWT) AS mat,
    SUM(W_FSTUWT * oku) / SUM(W_FSTUWT) AS oku,
    SUM(W_FSTUWT * fen) / SUM(W_FSTUWT) AS fen
  FROM uzun
  GROUP BY CYCLE, CNT, pv_no
),
ulke AS (                  -- 10 sonucun ortalaması (PV'lerin ortalaması DEĞİL)
  SELECT CYCLE, CNT, ANY_VALUE(oecd) AS oecd,
    AVG(mat) AS mat, AVG(oku) AS oku, AVG(fen) AS fen
  FROM pv_ort
  GROUP BY CYCLE, CNT
)
SELECT * FROM (
  SELECT CNT AS ulke, ROUND(mat, 1) AS matematik, ROUND(oku, 1) AS okuma, ROUND(fen, 1) AS fen
  FROM ulke
  UNION ALL
  SELECT 'OECD ortalaması', ROUND(AVG(mat), 1), ROUND(AVG(oku), 1), ROUND(AVG(fen), 1)
  FROM ulke
  WHERE oecd = 1
)
ORDER BY matematik DESC
""")

✔ 0.0 MB tarandı · 0.1 sn · önbellek: evet


,ulke,matematik,okuma,fen
0,QCI,612.4,526.5,597.0
1,SGP,562.9,534.9,559.9
2,MAC,549.3,501.1,540.8
3,TAP,546.2,507.8,539.7
4,JPN,525.4,502.7,537.6
...,...,...,...,...
86,PRY,333.7,352.2,355.2
87,KEN,325.9,320.2,334.9
88,ZMB,313.6,324.9,337.0
89,RWA,311.6,300.9,317.3


---
## Slayt 20 · Sosyoekonomik gruplara göre fen
`sql/04_escs_gruplari.sql`

> 04 · Sosyoekonomik gruplara göre fen başarısı (Türkiye, tüm döngüler)
> Slayt: "Sosyoekonomik gruplara göre fen başarısı"
> ESCS: OECD ortalaması 0, standart sapması 1 olacak biçimde ölçeklenir.

In [ ]:
calistir(r"""
WITH taban AS (
  SELECT
    CYCLE,
    W_FSTUWT AS w,
    CASE
      WHEN ESCS < -1 THEN '1 Düşük (ESCS < -1)'
      WHEN ESCS <  0 THEN '2 Orta-alt'
      WHEN ESCS <  1 THEN '3 Orta-üst'
      ELSE                '4 Yüksek (ESCS ≥ 1)'
    END AS escs_grup,
    PV1SCIE, PV2SCIE, PV3SCIE, PV4SCIE, PV5SCIE,
    PV6SCIE, PV7SCIE, PV8SCIE, PV9SCIE, PV10SCIE
  FROM `pisa.ogrenci`
  WHERE CNT = 'TUR' AND ESCS IS NOT NULL
),
uzun AS (
  SELECT * FROM taban
  UNPIVOT (puan FOR pv_no IN (
    PV1SCIE, PV2SCIE, PV3SCIE, PV4SCIE, PV5SCIE,
    PV6SCIE, PV7SCIE, PV8SCIE, PV9SCIE, PV10SCIE))
),
pv_ort AS (
  SELECT CYCLE, escs_grup, pv_no,
         SUM(w * puan) / SUM(w) AS ort,
         SUM(w) AS agirlik
  FROM uzun
  GROUP BY CYCLE, escs_grup, pv_no
)
SELECT
  CYCLE AS dongu,
  escs_grup,
  ROUND(AVG(ort), 1) AS fen,
  ROUND(100 * AVG(agirlik) / SUM(AVG(agirlik)) OVER (PARTITION BY CYCLE), 1) AS ogrenci_yuzdesi
FROM pv_ort
GROUP BY dongu, escs_grup
ORDER BY escs_grup, dongu
""")

✔ 0.0 MB tarandı · 0.1 sn · önbellek: evet


,dongu,escs_grup,fen,ogrenci_yuzdesi
0,2025,1 Düşük (ESCS < -1),475.3,54.6
1,2025,2 Orta-alt,500.6,22.6
2,2025,3 Orta-üst,546.3,19.2
3,2025,4 Yüksek (ESCS ≥ 1),547.9,3.6


---
## Slayt 21 · Doğru standart hata: 810 tahmin
`sql/03_standart_hata.sql`

> 03 · Doğru standart hata: 10 PV × 81 ağırlık = ülke başına 810 tahmin
> Slayt: "Ağırlıklar ve olası değerler: dört adım"
>   Örnekleme varyansı (Fay BRR, k = 0,5): U = (1/20) · Σ (θ_r − θ)²
>   Ölçme varyansı (PV'ler arası):        B = (1/9)  · Σ (θ_i − θ̄)²
>   Toplam: V = Ū + (1 + 1/10) · B     →  SH = √V
> Matematik yerine fen için: PV…MATH → PV…SCIE
> Daha hızlı demo için WHERE satırına ülke listesi ekleyin:
>   AND CNT IN ('TUR', 'GRC', 'DEU', 'JPN', 'SGP')

Canlı hesaplamayı göstermek için önbellek kapalı çalışır (`onbellek=False`).

In [ ]:
calistir(r"""
WITH taban AS (
  SELECT
    CNT,
    [PV1MATH, PV2MATH, PV3MATH, PV4MATH, PV5MATH,
     PV6MATH, PV7MATH, PV8MATH, PV9MATH, PV10MATH] AS pvler,
    [W_FSTUWT, W_FSTURWT1, W_FSTURWT2, W_FSTURWT3, W_FSTURWT4, W_FSTURWT5, W_FSTURWT6, W_FSTURWT7,
     W_FSTURWT8, W_FSTURWT9, W_FSTURWT10, W_FSTURWT11, W_FSTURWT12, W_FSTURWT13, W_FSTURWT14, W_FSTURWT15,
     W_FSTURWT16, W_FSTURWT17, W_FSTURWT18, W_FSTURWT19, W_FSTURWT20, W_FSTURWT21, W_FSTURWT22, W_FSTURWT23,
     W_FSTURWT24, W_FSTURWT25, W_FSTURWT26, W_FSTURWT27, W_FSTURWT28, W_FSTURWT29, W_FSTURWT30, W_FSTURWT31,
     W_FSTURWT32, W_FSTURWT33, W_FSTURWT34, W_FSTURWT35, W_FSTURWT36, W_FSTURWT37, W_FSTURWT38, W_FSTURWT39,
     W_FSTURWT40, W_FSTURWT41, W_FSTURWT42, W_FSTURWT43, W_FSTURWT44, W_FSTURWT45, W_FSTURWT46, W_FSTURWT47,
     W_FSTURWT48, W_FSTURWT49, W_FSTURWT50, W_FSTURWT51, W_FSTURWT52, W_FSTURWT53, W_FSTURWT54, W_FSTURWT55,
     W_FSTURWT56, W_FSTURWT57, W_FSTURWT58, W_FSTURWT59, W_FSTURWT60, W_FSTURWT61, W_FSTURWT62, W_FSTURWT63,
     W_FSTURWT64, W_FSTURWT65, W_FSTURWT66, W_FSTURWT67, W_FSTURWT68, W_FSTURWT69, W_FSTURWT70, W_FSTURWT71,
     W_FSTURWT72, W_FSTURWT73, W_FSTURWT74, W_FSTURWT75, W_FSTURWT76, W_FSTURWT77, W_FSTURWT78, W_FSTURWT79,
     W_FSTURWT80] AS agirliklar
  FROM `pisa.ogrenci`
  WHERE CYCLE = 2025
),
tahmin AS (            -- her ülke × PV (0–9) × ağırlık (0 = tam, 1–80 = tekrarlı)
  SELECT CNT, p AS pv_no, r AS agirlik_no,
         SUM(w * puan) / SUM(w) AS theta
  FROM taban,
       UNNEST(pvler) AS puan WITH OFFSET p,
       UNNEST(agirliklar) AS w WITH OFFSET r
  GROUP BY CNT, pv_no, agirlik_no
),
tam AS (               -- tam ağırlıkla 10 tahmin
  SELECT CNT, pv_no, theta AS theta_tam
  FROM tahmin
  WHERE agirlik_no = 0
),
orneklem AS (          -- her PV için örnekleme varyansı (Fay: 1 / (80 · 0,5²) = 1/20)
  SELECT CNT, pv_no, SUM(POW(t.theta - k.theta_tam, 2)) / 20 AS u
  FROM tahmin AS t
  JOIN tam AS k USING (CNT, pv_no)
  WHERE t.agirlik_no > 0
  GROUP BY CNT, pv_no
)
SELECT
  CNT AS ulke,
  ROUND(AVG(theta_tam), 1)                                      AS ortalama,
  ROUND(SQRT(AVG(u) + (1 + 1 / 10) * VAR_SAMP(theta_tam)), 2)   AS standart_hata,
  ROUND(SQRT(AVG(u)), 2)                                        AS sh_yalniz_orneklem,
  ROUND(SQRT((1 + 1 / 10) * VAR_SAMP(theta_tam)), 2)            AS sh_yalniz_olcme
FROM tam
JOIN orneklem USING (CNT, pv_no)
GROUP BY ulke
ORDER BY ortalama DESC
""", onbellek=False)

✔ 559.3 MB tarandı · 40.1 sn · önbellek: hayır


,ulke,ortalama,standart_hata,sh_yalniz_orneklem,sh_yalniz_olcme
0,QCI,612.4,2.55,2.52,0.33
1,SGP,562.9,1.57,1.51,0.45
2,MAC,549.3,1.15,1.04,0.48
3,TAP,546.2,3.12,3.08,0.46
4,JPN,525.4,4.20,4.19,0.29
...,...,...,...,...,...
85,PRY,333.7,1.59,1.52,0.46
86,KEN,325.9,2.69,2.68,0.28
87,ZMB,313.6,9.27,9.27,0.29
88,RWA,311.6,3.03,3.02,0.21


---
## Slayt 22 · Yeterlik düzeyleri
`sql/05_yeterlik_duzeyleri.sql`

> 05 · Yeterlik düzeyleri: Düzey 2 altı ve Düzey 5 ve üstü öğrenci yüzdesi
> Slayt: "PISA 2025: Türkiye, OECD ve tüm ülkeler"
> DOĞRULAMA (MEB, PISA 2025): Türkiye fen Düzey 5+ ≈ %7,4 (OECD ≈ %7,2),
>                             okuma Düzey 5+ ≈ %4,2
> EŞİKLER (önceki döngülerin ölçek eşikleri):
>   Matematik: Düzey 2 = 420,07 · Düzey 5 = 606,99
>   Okuma    : Düzey 2 = 407,47 · Düzey 5 = 625,61
>   Fen      : Düzey 2 = 409,54 · Düzey 5 = 633,33
> PISA 2025'te fen çerçevesi yenilendi: fen eşiklerini 2025 teknik raporundan
> teyit edin. Yukarıdaki %7,4 tutuyorsa eşikler doğrudur.

In [ ]:
calistir(r"""
WITH secim AS (
  SELECT CNT, OECD, W_FSTUWT,
    PV1MATH, PV2MATH, PV3MATH, PV4MATH, PV5MATH,
    PV6MATH, PV7MATH, PV8MATH, PV9MATH, PV10MATH,
    PV1READ, PV2READ, PV3READ, PV4READ, PV5READ,
    PV6READ, PV7READ, PV8READ, PV9READ, PV10READ,
    PV1SCIE, PV2SCIE, PV3SCIE, PV4SCIE, PV5SCIE,
    PV6SCIE, PV7SCIE, PV8SCIE, PV9SCIE, PV10SCIE
  FROM `pisa.ogrenci`
  WHERE CYCLE = 2025
),
uzun AS (
  SELECT * FROM secim
  UNPIVOT ((mat, oku, fen) FOR pv_no IN (
      (PV1MATH, PV1READ, PV1SCIE) AS 1,
      (PV2MATH, PV2READ, PV2SCIE) AS 2,
      (PV3MATH, PV3READ, PV3SCIE) AS 3,
      (PV4MATH, PV4READ, PV4SCIE) AS 4,
      (PV5MATH, PV5READ, PV5SCIE) AS 5,
      (PV6MATH, PV6READ, PV6SCIE) AS 6,
      (PV7MATH, PV7READ, PV7SCIE) AS 7,
      (PV8MATH, PV8READ, PV8SCIE) AS 8,
      (PV9MATH, PV9READ, PV9SCIE) AS 9,
      (PV10MATH, PV10READ, PV10SCIE) AS 10))
),
pv_oran AS (             -- her ülke × PV için ağırlıklı oranlar
  SELECT CNT, ANY_VALUE(OECD) AS oecd, pv_no,
    SUM(IF(mat <  420.07, W_FSTUWT, 0)) / SUM(W_FSTUWT) AS mat_alt2,
    SUM(IF(mat >= 606.99, W_FSTUWT, 0)) / SUM(W_FSTUWT) AS mat_ust5,
    SUM(IF(oku <  407.47, W_FSTUWT, 0)) / SUM(W_FSTUWT) AS oku_alt2,
    SUM(IF(oku >= 625.61, W_FSTUWT, 0)) / SUM(W_FSTUWT) AS oku_ust5,
    SUM(IF(fen <  409.54, W_FSTUWT, 0)) / SUM(W_FSTUWT) AS fen_alt2,
    SUM(IF(fen >= 633.33, W_FSTUWT, 0)) / SUM(W_FSTUWT) AS fen_ust5
  FROM uzun
  GROUP BY CNT, pv_no
),
ulke AS (                -- 10 PV sonucunun ortalaması
  SELECT CNT, ANY_VALUE(oecd) AS oecd,
    AVG(mat_alt2) AS mat_alt2, AVG(mat_ust5) AS mat_ust5,
    AVG(oku_alt2) AS oku_alt2, AVG(oku_ust5) AS oku_ust5,
    AVG(fen_alt2) AS fen_alt2, AVG(fen_ust5) AS fen_ust5
  FROM pv_oran
  GROUP BY CNT
)
SELECT * FROM (
  SELECT CNT AS ulke,
    ROUND(100 * fen_alt2, 1) AS fen_duzey2_alti, ROUND(100 * fen_ust5, 1) AS fen_duzey5_ustu,
    ROUND(100 * oku_alt2, 1) AS okuma_duzey2_alti, ROUND(100 * oku_ust5, 1) AS okuma_duzey5_ustu,
    ROUND(100 * mat_alt2, 1) AS mat_duzey2_alti, ROUND(100 * mat_ust5, 1) AS mat_duzey5_ustu
  FROM ulke
  UNION ALL
  SELECT 'OECD ortalaması',
    ROUND(100 * AVG(fen_alt2), 1), ROUND(100 * AVG(fen_ust5), 1),
    ROUND(100 * AVG(oku_alt2), 1), ROUND(100 * AVG(oku_ust5), 1),
    ROUND(100 * AVG(mat_alt2), 1), ROUND(100 * AVG(mat_ust5), 1)
  FROM ulke WHERE oecd = 1
)
WHERE ulke IN ('TUR', 'OECD ortalaması')   -- tüm ülkeler için bu satırı silin
ORDER BY ulke DESC
""")

✔ 0.0 MB tarandı · 0.1 sn · önbellek: evet


,ulke,fen_duzey2_alti,fen_duzey5_ustu,okuma_duzey2_alti,okuma_duzey5_ustu,mat_duzey2_alti,mat_duzey5_ustu
0,TUR,19.5,7.4,24.1,4.2,35.7,7.9
1,OECD ortalaması,25.7,7.2,31.0,5.5,34.9,7.8


---
## Slayt 25 · Türkiye: cinsiyet farkı
`sql/10_turkiye_cinsiyet.sql`

> 10 · Türkiye: cinsiyet farkı, üç alanda, standart hata ve anlamlılıkla
> Slayt: "Türkiye derinlemesine: cinsiyet"
> ST004D01T: 1 = kız, 2 = erkek (PISA 2022 kodlaması; 2025'te teyit edin)
> Fark = kız − erkek. |fark| > 1,96 × SH ise %5 düzeyinde anlamlı.

In [ ]:
calistir(r"""
WITH taban AS (
  SELECT CYCLE, CAST(ST004D01T AS INT64) AS cins, d.alan, d.pvler,
    [W_FSTUWT, W_FSTURWT1, W_FSTURWT2, W_FSTURWT3, W_FSTURWT4, W_FSTURWT5, W_FSTURWT6, W_FSTURWT7,
     W_FSTURWT8, W_FSTURWT9, W_FSTURWT10, W_FSTURWT11, W_FSTURWT12, W_FSTURWT13, W_FSTURWT14, W_FSTURWT15,
     W_FSTURWT16, W_FSTURWT17, W_FSTURWT18, W_FSTURWT19, W_FSTURWT20, W_FSTURWT21, W_FSTURWT22, W_FSTURWT23,
     W_FSTURWT24, W_FSTURWT25, W_FSTURWT26, W_FSTURWT27, W_FSTURWT28, W_FSTURWT29, W_FSTURWT30, W_FSTURWT31,
     W_FSTURWT32, W_FSTURWT33, W_FSTURWT34, W_FSTURWT35, W_FSTURWT36, W_FSTURWT37, W_FSTURWT38, W_FSTURWT39,
     W_FSTURWT40, W_FSTURWT41, W_FSTURWT42, W_FSTURWT43, W_FSTURWT44, W_FSTURWT45, W_FSTURWT46, W_FSTURWT47,
     W_FSTURWT48, W_FSTURWT49, W_FSTURWT50, W_FSTURWT51, W_FSTURWT52, W_FSTURWT53, W_FSTURWT54, W_FSTURWT55,
     W_FSTURWT56, W_FSTURWT57, W_FSTURWT58, W_FSTURWT59, W_FSTURWT60, W_FSTURWT61, W_FSTURWT62, W_FSTURWT63,
     W_FSTURWT64, W_FSTURWT65, W_FSTURWT66, W_FSTURWT67, W_FSTURWT68, W_FSTURWT69, W_FSTURWT70, W_FSTURWT71,
     W_FSTURWT72, W_FSTURWT73, W_FSTURWT74, W_FSTURWT75, W_FSTURWT76, W_FSTURWT77, W_FSTURWT78, W_FSTURWT79,
     W_FSTURWT80] AS agirliklar
  FROM `pisa.ogrenci`,
  UNNEST([
    STRUCT('Matematik' AS alan, [PV1MATH, PV2MATH, PV3MATH, PV4MATH, PV5MATH,
     PV6MATH, PV7MATH, PV8MATH, PV9MATH, PV10MATH] AS pvler),
    STRUCT('Okuma', [PV1READ, PV2READ, PV3READ, PV4READ, PV5READ,
     PV6READ, PV7READ, PV8READ, PV9READ, PV10READ]),
    STRUCT('Fen', [PV1SCIE, PV2SCIE, PV3SCIE, PV4SCIE, PV5SCIE,
     PV6SCIE, PV7SCIE, PV8SCIE, PV9SCIE, PV10SCIE])
  ]) AS d
  WHERE CNT = 'TUR' AND ST004D01T IN (1, 2)
),
tahmin AS (
  SELECT CYCLE, alan, cins, p AS pv_no, r AS agirlik_no, SUM(w * puan) / SUM(w) AS theta
  FROM taban,
       UNNEST(pvler) AS puan WITH OFFSET p,
       UNNEST(agirliklar) AS w WITH OFFSET r
  GROUP BY CYCLE, alan, cins, pv_no, agirlik_no
),
fark AS (             -- her PV × ağırlık için kız − erkek farkı
  SELECT CYCLE, alan, pv_no, agirlik_no,
    MAX(IF(cins = 1, theta, NULL)) AS kiz,
    MAX(IF(cins = 2, theta, NULL)) AS erkek,
    MAX(IF(cins = 1, theta, NULL)) - MAX(IF(cins = 2, theta, NULL)) AS theta
  FROM tahmin
  GROUP BY CYCLE, alan, pv_no, agirlik_no
),
tam AS (
  SELECT CYCLE, alan, pv_no, kiz, erkek, theta AS theta_tam FROM fark WHERE agirlik_no = 0
),
orneklem AS (
  SELECT CYCLE, alan, pv_no, SUM(POW(f.theta - k.theta_tam, 2)) / 20 AS u
  FROM fark AS f JOIN tam AS k USING (CYCLE, alan, pv_no)
  WHERE f.agirlik_no > 0
  GROUP BY CYCLE, alan, pv_no
),
sonuc AS (
  SELECT CYCLE, alan, AVG(kiz) AS kiz, AVG(erkek) AS erkek, AVG(theta_tam) AS fark,
    SQRT(AVG(u) + (1 + 1 / 10) * VAR_SAMP(theta_tam)) AS sh
  FROM tam JOIN orneklem USING (CYCLE, alan, pv_no)
  GROUP BY CYCLE, alan
)
SELECT
  CYCLE AS dongu, alan,
  ROUND(kiz, 1) AS kiz, ROUND(erkek, 1) AS erkek,
  ROUND(fark, 1) AS fark_kiz_eksi_erkek,
  ROUND(sh, 1) AS standart_hata,
  IF(ABS(fark) > 1.96 * sh, 'anlamlı', 'anlamlı değil') AS p_005
FROM sonuc
ORDER BY alan, dongu
""")

✔ 0.0 MB tarandı · 0.2 sn · önbellek: evet


,dongu,alan,kiz,erkek,fark_kiz_eksi_erkek,standart_hata,p_005
0,2025,Fen,492.4,496.5,-4.0,5.0,anlamlı değil
1,2025,Matematik,453.4,471.2,-17.8,5.1,anlamlı
2,2025,Okuma,482.1,460.3,21.8,4.8,anlamlı


---
## Slayt 26 · Türkiye: okul türü ve tabaka
`sql/11_turkiye_program_bolge.sql`

> 11 · Türkiye: öğrenim programı (okul türü) ve örneklem tabakası (bölge)
> Slayt: "Türkiye derinlemesine: okul türü ve bölge"
> ÖNEMLİ VERİ NOTU:
>  · Kamuya açık PISA dosyasında İL bilgisi yoktur; örneklem il düzeyinde
>    temsili de değildir. Bölge karşılaştırması ancak örneklem tabakaları
>    (STRATUM) bölge içeriyorsa yapılabilir: etiketleri kontrol edin.
>  · PROGN = ulusal öğrenim programı (ör. Anadolu lisesi, meslek lisesi …).
>  · Etiket sütunları (…_ETIKET) yükleme defterinde SPSS değer etiketlerinden üretilir.
>  · "dikkat" uyarılı satırları yorumlamayın: az okul = büyük belirsizlik.
> Matematik yerine fen: grup_sorgusu içindeki PV…MATH → PV…SCIE

**A) Öğrenim programına göre matematik**

In [ ]:
calistir(r"""
WITH taban AS (
  SELECT CYCLE, COALESCE(CAST(COALESCE(PROGN_ETIKET, CAST(PROGN AS STRING)) AS STRING), '(bilinmiyor)') AS grup,
    [PV1MATH, PV2MATH, PV3MATH, PV4MATH, PV5MATH,
     PV6MATH, PV7MATH, PV8MATH, PV9MATH, PV10MATH] AS pvler,
    [W_FSTUWT, W_FSTURWT1, W_FSTURWT2, W_FSTURWT3, W_FSTURWT4, W_FSTURWT5, W_FSTURWT6, W_FSTURWT7,
     W_FSTURWT8, W_FSTURWT9, W_FSTURWT10, W_FSTURWT11, W_FSTURWT12, W_FSTURWT13, W_FSTURWT14, W_FSTURWT15,
     W_FSTURWT16, W_FSTURWT17, W_FSTURWT18, W_FSTURWT19, W_FSTURWT20, W_FSTURWT21, W_FSTURWT22, W_FSTURWT23,
     W_FSTURWT24, W_FSTURWT25, W_FSTURWT26, W_FSTURWT27, W_FSTURWT28, W_FSTURWT29, W_FSTURWT30, W_FSTURWT31,
     W_FSTURWT32, W_FSTURWT33, W_FSTURWT34, W_FSTURWT35, W_FSTURWT36, W_FSTURWT37, W_FSTURWT38, W_FSTURWT39,
     W_FSTURWT40, W_FSTURWT41, W_FSTURWT42, W_FSTURWT43, W_FSTURWT44, W_FSTURWT45, W_FSTURWT46, W_FSTURWT47,
     W_FSTURWT48, W_FSTURWT49, W_FSTURWT50, W_FSTURWT51, W_FSTURWT52, W_FSTURWT53, W_FSTURWT54, W_FSTURWT55,
     W_FSTURWT56, W_FSTURWT57, W_FSTURWT58, W_FSTURWT59, W_FSTURWT60, W_FSTURWT61, W_FSTURWT62, W_FSTURWT63,
     W_FSTURWT64, W_FSTURWT65, W_FSTURWT66, W_FSTURWT67, W_FSTURWT68, W_FSTURWT69, W_FSTURWT70, W_FSTURWT71,
     W_FSTURWT72, W_FSTURWT73, W_FSTURWT74, W_FSTURWT75, W_FSTURWT76, W_FSTURWT77, W_FSTURWT78, W_FSTURWT79,
     W_FSTURWT80] AS agirliklar
  FROM `pisa.ogrenci`
  WHERE CNT = 'TUR'
),
sayim AS (
  SELECT CYCLE, COALESCE(CAST(COALESCE(PROGN_ETIKET, CAST(PROGN AS STRING)) AS STRING), '(bilinmiyor)') AS grup,
    COUNT(*) AS ogrenci, COUNT(DISTINCT CNTSCHID) AS okul, SUM(W_FSTUWT) AS agirlik
  FROM `pisa.ogrenci`
  WHERE CNT = 'TUR'
  GROUP BY CYCLE, grup
),
tahmin AS (          -- her grup × PV × ağırlık için ağırlıklı ortalama
  SELECT CYCLE, grup, p AS pv_no, r AS agirlik_no, SUM(w * puan) / SUM(w) AS theta
  FROM taban,
       UNNEST(pvler) AS puan WITH OFFSET p,
       UNNEST(agirliklar) AS w WITH OFFSET r
  GROUP BY CYCLE, grup, pv_no, agirlik_no
),
tam AS (
  SELECT CYCLE, grup, pv_no, theta AS theta_tam FROM tahmin WHERE agirlik_no = 0
),
orneklem AS (         -- Fay BRR: 1/20
  SELECT CYCLE, grup, pv_no, SUM(POW(t.theta - k.theta_tam, 2)) / 20 AS u
  FROM tahmin AS t JOIN tam AS k USING (CYCLE, grup, pv_no)
  WHERE t.agirlik_no > 0
  GROUP BY CYCLE, grup, pv_no
),
sonuc AS (            -- Rubin: V = Ū + (1 + 1/10)·B
  SELECT CYCLE, grup,
    AVG(theta_tam) AS ortalama,
    SQRT(AVG(u) + (1 + 1 / 10) * VAR_SAMP(theta_tam)) AS sh
  FROM tam JOIN orneklem USING (CYCLE, grup, pv_no)
  GROUP BY CYCLE, grup
)
SELECT
  CYCLE AS dongu, grup,
  ROUND(ortalama, 1) AS ortalama,
  ROUND(sh, 1) AS standart_hata,
  ogrenci, okul,
  ROUND(100 * agirlik / SUM(agirlik) OVER (PARTITION BY CYCLE), 1) AS nufus_yuzdesi,
  IF(okul < 10, 'dikkat: 10 okuldan az', '') AS uyari
FROM sonuc JOIN sayim USING (CYCLE, grup)
ORDER BY dongu, ortalama DESC
""")

✔ 0.0 MB tarandı · 0.1 sn · önbellek: evet


,dongu,grup,ortalama,standart_hata,ogrenci,okul,nufus_yuzdesi,uyari
0,2025,Turkiye: Science High School,605.2,7.5,451,11,6.4,
1,2025,Turkiye: Social Sciences High School,549.2,11.3,248,6,1.1,dikkat: 10 okuldan az
2,2025,Turkiye: Multi-programme Anatolian High School...,510.0,5.0,40,1,0.7,dikkat: 10 okuldan az
3,2025,Turkiye: Anatolian High School,479.3,2.6,3586,93,50.8,
4,2025,Turkiye: Anatolian Imam Hatip High School,457.1,10.0,733,23,9.5,
5,2025,Turkiye: Multi-programme Anatolian High School...,423.4,5.7,40,1,0.6,dikkat: 10 okuldan az
6,2025,Turkiye: Anatolian Fine Arts High School,421.5,5.0,32,1,0.5,dikkat: 10 okuldan az
7,2025,Turkiye: Anatolian Sports High School,414.8,5.7,40,1,0.7,dikkat: 10 okuldan az
8,2025,Turkiye: Vocational and Technical Anatolian Hi...,399.9,3.9,2526,64,29.7,
9,2025,Turkiye: Secondary School,369.0,23.4,6,2,0.1,dikkat: 10 okuldan az


**B) Örneklem tabakasına göre matematik**

In [ ]:
calistir(r"""
WITH taban AS (
  SELECT CYCLE, COALESCE(CAST(COALESCE(STRATUM_ETIKET, CAST(STRATUM AS STRING)) AS STRING), '(bilinmiyor)') AS grup,
    [PV1MATH, PV2MATH, PV3MATH, PV4MATH, PV5MATH,
     PV6MATH, PV7MATH, PV8MATH, PV9MATH, PV10MATH] AS pvler,
    [W_FSTUWT, W_FSTURWT1, W_FSTURWT2, W_FSTURWT3, W_FSTURWT4, W_FSTURWT5, W_FSTURWT6, W_FSTURWT7,
     W_FSTURWT8, W_FSTURWT9, W_FSTURWT10, W_FSTURWT11, W_FSTURWT12, W_FSTURWT13, W_FSTURWT14, W_FSTURWT15,
     W_FSTURWT16, W_FSTURWT17, W_FSTURWT18, W_FSTURWT19, W_FSTURWT20, W_FSTURWT21, W_FSTURWT22, W_FSTURWT23,
     W_FSTURWT24, W_FSTURWT25, W_FSTURWT26, W_FSTURWT27, W_FSTURWT28, W_FSTURWT29, W_FSTURWT30, W_FSTURWT31,
     W_FSTURWT32, W_FSTURWT33, W_FSTURWT34, W_FSTURWT35, W_FSTURWT36, W_FSTURWT37, W_FSTURWT38, W_FSTURWT39,
     W_FSTURWT40, W_FSTURWT41, W_FSTURWT42, W_FSTURWT43, W_FSTURWT44, W_FSTURWT45, W_FSTURWT46, W_FSTURWT47,
     W_FSTURWT48, W_FSTURWT49, W_FSTURWT50, W_FSTURWT51, W_FSTURWT52, W_FSTURWT53, W_FSTURWT54, W_FSTURWT55,
     W_FSTURWT56, W_FSTURWT57, W_FSTURWT58, W_FSTURWT59, W_FSTURWT60, W_FSTURWT61, W_FSTURWT62, W_FSTURWT63,
     W_FSTURWT64, W_FSTURWT65, W_FSTURWT66, W_FSTURWT67, W_FSTURWT68, W_FSTURWT69, W_FSTURWT70, W_FSTURWT71,
     W_FSTURWT72, W_FSTURWT73, W_FSTURWT74, W_FSTURWT75, W_FSTURWT76, W_FSTURWT77, W_FSTURWT78, W_FSTURWT79,
     W_FSTURWT80] AS agirliklar
  FROM `pisa.ogrenci`
  WHERE CNT = 'TUR'
),
sayim AS (
  SELECT CYCLE, COALESCE(CAST(COALESCE(STRATUM_ETIKET, CAST(STRATUM AS STRING)) AS STRING), '(bilinmiyor)') AS grup,
    COUNT(*) AS ogrenci, COUNT(DISTINCT CNTSCHID) AS okul, SUM(W_FSTUWT) AS agirlik
  FROM `pisa.ogrenci`
  WHERE CNT = 'TUR'
  GROUP BY CYCLE, grup
),
tahmin AS (          -- her grup × PV × ağırlık için ağırlıklı ortalama
  SELECT CYCLE, grup, p AS pv_no, r AS agirlik_no, SUM(w * puan) / SUM(w) AS theta
  FROM taban,
       UNNEST(pvler) AS puan WITH OFFSET p,
       UNNEST(agirliklar) AS w WITH OFFSET r
  GROUP BY CYCLE, grup, pv_no, agirlik_no
),
tam AS (
  SELECT CYCLE, grup, pv_no, theta AS theta_tam FROM tahmin WHERE agirlik_no = 0
),
orneklem AS (         -- Fay BRR: 1/20
  SELECT CYCLE, grup, pv_no, SUM(POW(t.theta - k.theta_tam, 2)) / 20 AS u
  FROM tahmin AS t JOIN tam AS k USING (CYCLE, grup, pv_no)
  WHERE t.agirlik_no > 0
  GROUP BY CYCLE, grup, pv_no
),
sonuc AS (            -- Rubin: V = Ū + (1 + 1/10)·B
  SELECT CYCLE, grup,
    AVG(theta_tam) AS ortalama,
    SQRT(AVG(u) + (1 + 1 / 10) * VAR_SAMP(theta_tam)) AS sh
  FROM tam JOIN orneklem USING (CYCLE, grup, pv_no)
  GROUP BY CYCLE, grup
)
SELECT
  CYCLE AS dongu, grup,
  ROUND(ortalama, 1) AS ortalama,
  ROUND(sh, 1) AS standart_hata,
  ogrenci, okul,
  ROUND(100 * agirlik / SUM(agirlik) OVER (PARTITION BY CYCLE), 1) AS nufus_yuzdesi,
  IF(okul < 10, 'dikkat: 10 okuldan az', '') AS uyari
FROM sonuc JOIN sayim USING (CYCLE, grup)
ORDER BY dongu, ortalama DESC
""")

✔ 0.0 MB tarandı · 0.2 sn · önbellek: evet


,dongu,grup,ortalama,standart_hata,ogrenci,okul,nufus_yuzdesi,uyari
0,2025,Science High School - B,643.9,9.1,124,3,1.9,dikkat: 10 okuldan az
1,2025,Vocational and Technical Anatolian High School...,642.8,5.3,84,2,0.1,dikkat: 10 okuldan az
2,2025,Science High School - A,640.0,18.5,84,2,1.0,dikkat: 10 okuldan az
3,2025,Anatolian High School - A,634.6,13.2,82,2,0.7,dikkat: 10 okuldan az
4,2025,Private Vocational High School - A,629.7,39.4,83,2,0.2,dikkat: 10 okuldan az
5,2025,Anatolian Imam and Preacher High School - A,617.4,12.8,78,2,0.5,dikkat: 10 okuldan az
6,2025,Anatolian High School - B,598.4,5.4,126,3,2.1,dikkat: 10 okuldan az
7,2025,Anatolian High School - C,581.6,18.2,126,3,1.7,dikkat: 10 okuldan az
8,2025,Anatolian High School - D,581.6,9.3,126,3,1.8,dikkat: 10 okuldan az
9,2025,Science High School - C,578.9,19.6,126,3,1.6,dikkat: 10 okuldan az


---
## Slayt 26 · Türkiye: yerleşim yeri ve devlet/özel
`sql/13_okul_konum_turu.sql`

> 13 · Türkiye: okulun bulunduğu yerleşim yeri ve okul türü (devlet/özel)
> Slayt: "Türkiye derinlemesine: okullar"
> GEREKSİNİM: Yükleme defterinde OKUL dosyası da yüklenmiş olmalı (pisa.okul).
> SC001Q01TA: okulun bulunduğu yerleşim yeri büyüklüğü (köy … büyükşehir)
> SCHLTYPE  : okul türü (özel bağımsız / devlete bağlı özel / devlet)

**A) Yerleşim yerine göre matematik**

In [ ]:
calistir(r"""
WITH taban AS (
  SELECT CYCLE, COALESCE(CAST(k.SC001Q01TA_ETIKET AS STRING), '(bilinmiyor)') AS grup,
    [PV1MATH, PV2MATH, PV3MATH, PV4MATH, PV5MATH,
     PV6MATH, PV7MATH, PV8MATH, PV9MATH, PV10MATH] AS pvler,
    [W_FSTUWT, W_FSTURWT1, W_FSTURWT2, W_FSTURWT3, W_FSTURWT4, W_FSTURWT5, W_FSTURWT6, W_FSTURWT7,
     W_FSTURWT8, W_FSTURWT9, W_FSTURWT10, W_FSTURWT11, W_FSTURWT12, W_FSTURWT13, W_FSTURWT14, W_FSTURWT15,
     W_FSTURWT16, W_FSTURWT17, W_FSTURWT18, W_FSTURWT19, W_FSTURWT20, W_FSTURWT21, W_FSTURWT22, W_FSTURWT23,
     W_FSTURWT24, W_FSTURWT25, W_FSTURWT26, W_FSTURWT27, W_FSTURWT28, W_FSTURWT29, W_FSTURWT30, W_FSTURWT31,
     W_FSTURWT32, W_FSTURWT33, W_FSTURWT34, W_FSTURWT35, W_FSTURWT36, W_FSTURWT37, W_FSTURWT38, W_FSTURWT39,
     W_FSTURWT40, W_FSTURWT41, W_FSTURWT42, W_FSTURWT43, W_FSTURWT44, W_FSTURWT45, W_FSTURWT46, W_FSTURWT47,
     W_FSTURWT48, W_FSTURWT49, W_FSTURWT50, W_FSTURWT51, W_FSTURWT52, W_FSTURWT53, W_FSTURWT54, W_FSTURWT55,
     W_FSTURWT56, W_FSTURWT57, W_FSTURWT58, W_FSTURWT59, W_FSTURWT60, W_FSTURWT61, W_FSTURWT62, W_FSTURWT63,
     W_FSTURWT64, W_FSTURWT65, W_FSTURWT66, W_FSTURWT67, W_FSTURWT68, W_FSTURWT69, W_FSTURWT70, W_FSTURWT71,
     W_FSTURWT72, W_FSTURWT73, W_FSTURWT74, W_FSTURWT75, W_FSTURWT76, W_FSTURWT77, W_FSTURWT78, W_FSTURWT79,
     W_FSTURWT80] AS agirliklar
  FROM `pisa.ogrenci` AS o JOIN `pisa.okul` AS k USING (CYCLE, CNT, CNTSCHID)
  WHERE CNT = 'TUR'
),
sayim AS (
  SELECT CYCLE, COALESCE(CAST(k.SC001Q01TA_ETIKET AS STRING), '(bilinmiyor)') AS grup,
    COUNT(*) AS ogrenci, COUNT(DISTINCT CNTSCHID) AS okul, SUM(W_FSTUWT) AS agirlik
  FROM `pisa.ogrenci` AS o JOIN `pisa.okul` AS k USING (CYCLE, CNT, CNTSCHID)
  WHERE CNT = 'TUR'
  GROUP BY CYCLE, grup
),
tahmin AS (          -- her grup × PV × ağırlık için ağırlıklı ortalama
  SELECT CYCLE, grup, p AS pv_no, r AS agirlik_no, SUM(w * puan) / SUM(w) AS theta
  FROM taban,
       UNNEST(pvler) AS puan WITH OFFSET p,
       UNNEST(agirliklar) AS w WITH OFFSET r
  GROUP BY CYCLE, grup, pv_no, agirlik_no
),
tam AS (
  SELECT CYCLE, grup, pv_no, theta AS theta_tam FROM tahmin WHERE agirlik_no = 0
),
orneklem AS (         -- Fay BRR: 1/20
  SELECT CYCLE, grup, pv_no, SUM(POW(t.theta - k.theta_tam, 2)) / 20 AS u
  FROM tahmin AS t JOIN tam AS k USING (CYCLE, grup, pv_no)
  WHERE t.agirlik_no > 0
  GROUP BY CYCLE, grup, pv_no
),
sonuc AS (            -- Rubin: V = Ū + (1 + 1/10)·B
  SELECT CYCLE, grup,
    AVG(theta_tam) AS ortalama,
    SQRT(AVG(u) + (1 + 1 / 10) * VAR_SAMP(theta_tam)) AS sh
  FROM tam JOIN orneklem USING (CYCLE, grup, pv_no)
  GROUP BY CYCLE, grup
)
SELECT
  CYCLE AS dongu, grup,
  ROUND(ortalama, 1) AS ortalama,
  ROUND(sh, 1) AS standart_hata,
  ogrenci, okul,
  ROUND(100 * agirlik / SUM(agirlik) OVER (PARTITION BY CYCLE), 1) AS nufus_yuzdesi,
  IF(okul < 10, 'dikkat: 10 okuldan az', '') AS uyari
FROM sonuc JOIN sayim USING (CYCLE, grup)
ORDER BY dongu, ortalama DESC
""")

✔ 0.0 MB tarandı · 0.4 sn · önbellek: evet


,dongu,grup,ortalama,standart_hata,ogrenci,okul,nufus_yuzdesi,uyari
0,2025,A megacity (with over 10 000 000 people),476.8,17.1,972,25,13.2,
1,2025,A large city (1 000 000 to about 10 000 000 pe...,473.1,8.1,2648,69,34.8,
2,2025,A city (100 000 to about 1 000 000 people),453.7,10.5,1825,47,22.3,
3,2025,A town (15 000 to about 100 000 people),451.6,10.9,1915,51,25.4,
4,2025,"A village, hamlet or rural area (fewer than 3 ...",444.6,46.9,82,2,0.8,dikkat: 10 okuldan az
5,2025,(bilinmiyor),432.9,47.7,97,3,1.2,dikkat: 10 okuldan az
6,2025,A small town (3 000 to about 15 000 people),404.7,24.0,163,6,2.3,dikkat: 10 okuldan az


**B) Okul türüne göre matematik**

In [ ]:
calistir(r"""
WITH taban AS (
  SELECT CYCLE, COALESCE(CAST(k.SCHLTYPE_ETIKET AS STRING), '(bilinmiyor)') AS grup,
    [PV1MATH, PV2MATH, PV3MATH, PV4MATH, PV5MATH,
     PV6MATH, PV7MATH, PV8MATH, PV9MATH, PV10MATH] AS pvler,
    [W_FSTUWT, W_FSTURWT1, W_FSTURWT2, W_FSTURWT3, W_FSTURWT4, W_FSTURWT5, W_FSTURWT6, W_FSTURWT7,
     W_FSTURWT8, W_FSTURWT9, W_FSTURWT10, W_FSTURWT11, W_FSTURWT12, W_FSTURWT13, W_FSTURWT14, W_FSTURWT15,
     W_FSTURWT16, W_FSTURWT17, W_FSTURWT18, W_FSTURWT19, W_FSTURWT20, W_FSTURWT21, W_FSTURWT22, W_FSTURWT23,
     W_FSTURWT24, W_FSTURWT25, W_FSTURWT26, W_FSTURWT27, W_FSTURWT28, W_FSTURWT29, W_FSTURWT30, W_FSTURWT31,
     W_FSTURWT32, W_FSTURWT33, W_FSTURWT34, W_FSTURWT35, W_FSTURWT36, W_FSTURWT37, W_FSTURWT38, W_FSTURWT39,
     W_FSTURWT40, W_FSTURWT41, W_FSTURWT42, W_FSTURWT43, W_FSTURWT44, W_FSTURWT45, W_FSTURWT46, W_FSTURWT47,
     W_FSTURWT48, W_FSTURWT49, W_FSTURWT50, W_FSTURWT51, W_FSTURWT52, W_FSTURWT53, W_FSTURWT54, W_FSTURWT55,
     W_FSTURWT56, W_FSTURWT57, W_FSTURWT58, W_FSTURWT59, W_FSTURWT60, W_FSTURWT61, W_FSTURWT62, W_FSTURWT63,
     W_FSTURWT64, W_FSTURWT65, W_FSTURWT66, W_FSTURWT67, W_FSTURWT68, W_FSTURWT69, W_FSTURWT70, W_FSTURWT71,
     W_FSTURWT72, W_FSTURWT73, W_FSTURWT74, W_FSTURWT75, W_FSTURWT76, W_FSTURWT77, W_FSTURWT78, W_FSTURWT79,
     W_FSTURWT80] AS agirliklar
  FROM `pisa.ogrenci` AS o JOIN `pisa.okul` AS k USING (CYCLE, CNT, CNTSCHID)
  WHERE CNT = 'TUR'
),
sayim AS (
  SELECT CYCLE, COALESCE(CAST(k.SCHLTYPE_ETIKET AS STRING), '(bilinmiyor)') AS grup,
    COUNT(*) AS ogrenci, COUNT(DISTINCT CNTSCHID) AS okul, SUM(W_FSTUWT) AS agirlik
  FROM `pisa.ogrenci` AS o JOIN `pisa.okul` AS k USING (CYCLE, CNT, CNTSCHID)
  WHERE CNT = 'TUR'
  GROUP BY CYCLE, grup
),
tahmin AS (          -- her grup × PV × ağırlık için ağırlıklı ortalama
  SELECT CYCLE, grup, p AS pv_no, r AS agirlik_no, SUM(w * puan) / SUM(w) AS theta
  FROM taban,
       UNNEST(pvler) AS puan WITH OFFSET p,
       UNNEST(agirliklar) AS w WITH OFFSET r
  GROUP BY CYCLE, grup, pv_no, agirlik_no
),
tam AS (
  SELECT CYCLE, grup, pv_no, theta AS theta_tam FROM tahmin WHERE agirlik_no = 0
),
orneklem AS (         -- Fay BRR: 1/20
  SELECT CYCLE, grup, pv_no, SUM(POW(t.theta - k.theta_tam, 2)) / 20 AS u
  FROM tahmin AS t JOIN tam AS k USING (CYCLE, grup, pv_no)
  WHERE t.agirlik_no > 0
  GROUP BY CYCLE, grup, pv_no
),
sonuc AS (            -- Rubin: V = Ū + (1 + 1/10)·B
  SELECT CYCLE, grup,
    AVG(theta_tam) AS ortalama,
    SQRT(AVG(u) + (1 + 1 / 10) * VAR_SAMP(theta_tam)) AS sh
  FROM tam JOIN orneklem USING (CYCLE, grup, pv_no)
  GROUP BY CYCLE, grup
)
SELECT
  CYCLE AS dongu, grup,
  ROUND(ortalama, 1) AS ortalama,
  ROUND(sh, 1) AS standart_hata,
  ogrenci, okul,
  ROUND(100 * agirlik / SUM(agirlik) OVER (PARTITION BY CYCLE), 1) AS nufus_yuzdesi,
  IF(okul < 10, 'dikkat: 10 okuldan az', '') AS uyari
FROM sonuc JOIN sayim USING (CYCLE, grup)
ORDER BY dongu, ortalama DESC
""")

✔ 0.0 MB tarandı · 0.2 sn · önbellek: evet


,dongu,grup,ortalama,standart_hata,ogrenci,okul,nufus_yuzdesi,uyari
0,2025,Public,463.7,2.0,6889,178,89.0,
1,2025,Private independent,457.9,10.3,735,23,10.0,
2,2025,Private Government-dependent,318.2,3.9,78,2,1.1,dikkat: 10 okuldan az


---
## Slayt 27 · Okullar arası fark
`sql/12_okullar_arasi_fark.sql`

> 12 · Okullar arası fark: başarı farklılıklarının ne kadarı okullar arasında?
> Slayt: "Türkiye derinlemesine: okullar"
> Betimsel varyans ayrıştırması: okul ortalamalarının varyansı / toplam varyans.
> (OECD raporları çok düzeyli model kullanır; değerler yakın ama birebir değildir.)

**A) Ülkelere göre okullar arası varyans payı (10 PV ortalaması)**

In [ ]:
calistir(r"""
WITH secim AS (
  SELECT CNT, OECD, CNTSCHID, W_FSTUWT AS w,
    PV1MATH, PV2MATH, PV3MATH, PV4MATH, PV5MATH,
    PV6MATH, PV7MATH, PV8MATH, PV9MATH, PV10MATH
  FROM `pisa.ogrenci`
  WHERE CYCLE = 2025
),
uzun AS (
  SELECT * FROM secim
  UNPIVOT (puan FOR pv_no IN (
    PV1MATH, PV2MATH, PV3MATH, PV4MATH, PV5MATH,
    PV6MATH, PV7MATH, PV8MATH, PV9MATH, PV10MATH))
),
genel AS (
  SELECT CNT, pv_no, SUM(w * puan) / SUM(w) AS m FROM uzun GROUP BY CNT, pv_no
),
okul AS (
  SELECT CNT, pv_no, CNTSCHID, SUM(w) AS wo, SUM(w * puan) / SUM(w) AS mo
  FROM uzun GROUP BY CNT, pv_no, CNTSCHID
),
toplam AS (
  SELECT CNT, pv_no, SUM(u.w * POW(u.puan - g.m, 2)) / SUM(u.w) AS v_toplam
  FROM uzun AS u JOIN genel AS g USING (CNT, pv_no) GROUP BY CNT, pv_no
),
arasi AS (
  SELECT CNT, pv_no, SUM(o.wo * POW(o.mo - g.m, 2)) / SUM(o.wo) AS v_arasi
  FROM okul AS o JOIN genel AS g USING (CNT, pv_no) GROUP BY CNT, pv_no
),
ulke AS (
  SELECT CNT, AVG(v_arasi / v_toplam) AS pay
  FROM toplam JOIN arasi USING (CNT, pv_no) GROUP BY CNT
)
SELECT * FROM (
  SELECT CNT AS ulke, ROUND(100 * pay, 1) AS okullar_arasi_yuzde FROM ulke
  UNION ALL
  SELECT 'OECD ortalaması', ROUND(100 * AVG(pay), 1)
  FROM ulke WHERE CNT IN (SELECT DISTINCT CNT FROM secim WHERE OECD = 1)
)
WHERE ulke IN ('TUR', 'OECD ortalaması', 'FIN', 'DEU', 'JPN', 'GRC')   -- tüm ülkeler için silin
ORDER BY okullar_arasi_yuzde DESC
""")

✔ 0.0 MB tarandı · 0.2 sn · önbellek: evet


,ulke,okullar_arasi_yuzde
0,TUR,63.6
1,JPN,55.3
2,DEU,48.9
3,OECD ortalaması,34.4
4,GRC,33.1
5,FIN,11.5


**B) Türkiye: okulların sosyoekonomik düzeyine göre dört grup**

In [ ]:
calistir(r"""
WITH okul AS (
  SELECT CNTSCHID,
    SAFE_DIVIDE(SUM(W_FSTUWT * ESCS), SUM(IF(ESCS IS NULL, 0, W_FSTUWT))) AS okul_escs,
    SUM(W_FSTUWT * (PV1MATH + PV2MATH + PV3MATH + PV4MATH + PV5MATH + PV6MATH + PV7MATH + PV8MATH + PV9MATH + PV10MATH) / 10) / SUM(W_FSTUWT) AS okul_mat,
    SUM(W_FSTUWT) AS w
  FROM `pisa.ogrenci`
  WHERE CNT = 'TUR' AND CYCLE = 2025
  GROUP BY CNTSCHID
),
ceyrek AS (
  SELECT *, NTILE(4) OVER (ORDER BY okul_escs) AS c FROM okul WHERE okul_escs IS NOT NULL
)
SELECT
  CASE c WHEN 1 THEN '1 En dezavantajlı okullar' WHEN 2 THEN '2' WHEN 3 THEN '3'
         ELSE '4 En avantajlı okullar' END AS okul_grubu,
  COUNT(*) AS okul_sayisi,
  ROUND(SUM(w * okul_escs) / SUM(w), 2) AS ort_escs,
  ROUND(SUM(w * okul_mat) / SUM(w), 1) AS matematik
FROM ceyrek
GROUP BY okul_grubu
ORDER BY okul_grubu
""")

✔ 0.0 MB tarandı · 0.1 sn · önbellek: evet


,okul_grubu,okul_sayisi,ort_escs,matematik
0,1 En dezavantajlı okullar,51,-1.79,395.5
1,2,51,-1.39,430.7
2,3,51,-0.86,482.7
3,4 En avantajlı okullar,50,-0.00,537.8


**C) Okul ESCS ile okul başarısı arasındaki korelasyon (Türkiye)**

In [ ]:
calistir(r"""
SELECT ROUND(CORR(okul_escs, okul_mat), 2) AS korelasyon, COUNT(*) AS okul_sayisi
FROM (
  SELECT CNTSCHID,
    SAFE_DIVIDE(SUM(W_FSTUWT * ESCS), SUM(IF(ESCS IS NULL, 0, W_FSTUWT))) AS okul_escs,
    SUM(W_FSTUWT * (PV1MATH + PV2MATH + PV3MATH + PV4MATH + PV5MATH + PV6MATH + PV7MATH + PV8MATH + PV9MATH + PV10MATH) / 10) / SUM(W_FSTUWT) AS okul_mat
  FROM `pisa.ogrenci`
  WHERE CNT = 'TUR' AND CYCLE = 2025
  GROUP BY CNTSCHID
)
""")

✔ 0.0 MB tarandı · 0.1 sn · önbellek: evet


,korelasyon,okul_sayisi
0,0.73,203


---
## Slayt 30 · Risk modeli: değerlendirme ve açıklama
`sql/06_model_risk.sql`

> 06 · BigQuery ML: matematikte Düzey 2 altında kalma riski
> Slayt: "Düzey 2 altında kalma riskini tahmin" + "Model ne kadar iyi?"
> ÖNEMLİ: Modelleri DEMODAN ÖNCE eğitin (ağaç modeli birkaç dakika sürer).
>         Demoda yalnızca C, D, E bölümlerini çalıştırın.
> Etiket: PV1MATH < 420,07 (OECD Düzey 2 alt sınırı). Puan özellik olarak girmez.
> Bölme: öğrenci kimliğinin özetiyle sabit %20 test → iki model aynı testte.
> Sınırlılık: BigQuery ML bu modellerde örneklem ağırlığı kullanmaz;
>             sonuçlar keşifseldir, nüfus kestirimi değildir.
> Veri: PISA 2025, Türkiye.
> Özellik listesi (ESCS … cinsiyet) 2025 dosyasında aynı adlarla olmalı:
> defterin 4. hücresinde "EKSİK" çıkan değişkeni 06, 07, 14, 15'ten silin
> ya da 2025'teki karşılığıyla değiştirin.

**C) İki modeli aynı test kümesinde karşılaştır**

In [ ]:
calistir(r"""
SELECT 'lojistik' AS model, precision, recall, f1_score, roc_auc
FROM ML.EVALUATE(MODEL `pisa.m_risk_lojistik`)
UNION ALL
SELECT 'boosted_tree', precision, recall, f1_score, roc_auc
FROM ML.EVALUATE(MODEL `pisa.m_risk_2025`)
""")

✔ 0.0 MB tarandı · 0.1 sn · önbellek: evet


,model,precision,recall,f1_score,roc_auc
0,lojistik,0.619772,0.334016,0.434088,0.712441
1,boosted_tree,0.594249,0.381148,0.464419,0.717767


**D) Genel değişken önemi**

In [ ]:
calistir(r"""
SELECT * FROM ML.GLOBAL_EXPLAIN(MODEL `pisa.m_risk_2025`)
ORDER BY attribution DESC
""")

✔ 0.0 MB tarandı · 0.1 sn · önbellek: evet


,feature,attribution
0,ESCS,0.198870
1,REPEAT,0.153901
2,HOMEPOS,0.088425
3,BELONG,0.041159
4,cinsiyet,0.033802


**E) Öğrenci düzeyinde açıklama (5 örnek öğrenci)**

In [ ]:
calistir(r"""
SELECT *
FROM ML.EXPLAIN_PREDICT(
  MODEL `pisa.m_risk_2025`,
  (SELECT CNTSTUID, {OZELLIKLER}
   FROM `pisa.ogrenci`
   WHERE CNT = 'TUR' AND CYCLE = 2025
   LIMIT 5),
  STRUCT(3 AS top_k_features))
""")

✔ 22.1 MB tarandı · 0.3 sn · önbellek: hayır


,predicted_duzey2_alti,probability,top_feature_attributions,baseline_prediction_value,prediction_value,approximation_error,CNTSTUID,ESCS,HOMEPOS,BELONG,REPEAT,cinsiyet
0,1,0.725773,"[{'feature': 'REPEAT', 'attribution': 0.534966...",0.250671,0.986640,0.0,79200041,-2.251918,-3.068580,-1.335027,1.0,2
1,0,0.627313,"[{'feature': 'ESCS', 'attribution': -0.1885458...",0.749329,0.760353,0.0,79204212,-2.076997,-1.052001,-0.566096,0.0,2
2,1,0.509859,"[{'feature': 'ESCS', 'attribution': 0.19521443...",0.250671,0.519720,0.0,79204213,-2.408643,-2.399787,-1.406768,0.0,1
3,0,0.518924,"[{'feature': 'ESCS', 'attribution': -0.1769569...",0.749329,0.537866,0.0,79204215,-1.854787,-2.327878,-0.820741,0.0,1
4,0,0.701715,"[{'feature': 'REPEAT', 'attribution': 0.134621...",0.749329,0.927740,0.0,79204214,-0.709381,-0.455495,-0.785280,0.0,1


---
## Slayt 31 · Türkiye'de öğren, başka ülkelerde sına
`sql/07_ulkeler_arasi_aktarim.sql`

> 07 · "Türkiye'de öğren, başka ülkelerde sına"  (06'daki m_risk_2025 gerekli)
> Slayt: "Türkiye'de öğren, başka ülkelerde sına"
> Soru: Türkiye'de başarı riskini belirleyen ilişkiler başka eğitim
>       sistemlerinde de geçerli mi? Karşılaştırma ülkelerini değiştirebilirsiniz.
> A bölümünü DEMODAN ÖNCE çalıştırın (~3–5 dk).

**B) Türkiye modeli başka ülkelerde ne kadar iyi?**

In [ ]:
calistir(karsi_sinama_sql())

✔ 43.1 MB tarandı · 0.6 sn · önbellek: hayır


,sinama,precision,recall,f1_score,roc_auc
0,TUR modeli → TUR (test kümesi),0.594249,0.381148,0.464419,0.717767
1,TUR modeli → ITA,0.494681,0.214698,0.299437,0.655193
2,TUR modeli → GRC,0.783163,0.093826,0.167576,0.667232
3,TUR modeli → PRT,0.667939,0.306748,0.420420,0.705498


**C) Başarı riskinin belirleyicileri aynı mı? Türkiye ve Yunanistan modelleri yan yana**

In [ ]:
calistir(karsi_onem_sql())

✔ 0.0 MB tarandı · 0.2 sn · önbellek: hayır


,feature,onem_TUR,onem_GRC
0,ESCS,0.199,0.167
1,REPEAT,0.154,0.028
2,HOMEPOS,0.088,0.034
3,BELONG,0.041,0.025
4,cinsiyet,0.034,0.011


---
## Slayt 32 · Öğrenci profilleri
`sql/08_kmeans_profiller.sql`

> 08 · K-ortalamalar ile öğrenci profilleri (Türkiye)
> Slayt: "K-ortalamalar ile öğrenci profilleri"
> A) bölümünü DEMODAN ÖNCE çalıştırın (~2 dk).
> Küme sayısını seçmek için num_clusters = 3, 4, 5 deneyip
> C) bölümündeki davies_bouldin_index'e bakın (küçük = daha iyi).

**B) Profil merkezleri: her profilin ortalama özellikleri**

In [ ]:
calistir(r"""
SELECT *
FROM (
  SELECT centroid_id AS profil, feature, ROUND(numerical_value, 2) AS deger
  FROM ML.CENTROIDS(MODEL `pisa.m_profil_2025`)
)
PIVOT (ANY_VALUE(deger) FOR feature IN ({PROFIL_IN}))
ORDER BY profil
""")

✔ 0.0 MB tarandı · 0.3 sn · önbellek: hayır


,profil,ESCS,BELONG,HOMEPOS
0,1,0.47,-0.04,0.04
1,2,-2.15,-0.33,-2.19
2,3,-0.85,2.29,-0.90
3,4,-0.97,-0.07,-1.00


**C) Kümeleme kalitesi**

In [ ]:
calistir(r"""
SELECT * FROM ML.EVALUATE(MODEL `pisa.m_profil_2025`)
""")

✔ 0.0 MB tarandı · 0.2 sn · önbellek: hayır


,davies_bouldin_index,mean_squared_distance
0,1.200587,1.140984


**D) Profillerin matematik başarısı (10 PV, ağırlıklı) ve nüfustaki payı**

In [ ]:
calistir(r"""
WITH atama AS (
  SELECT CENTROID_ID AS profil, W_FSTUWT AS w,
    PV1MATH, PV2MATH, PV3MATH, PV4MATH, PV5MATH,
    PV6MATH, PV7MATH, PV8MATH, PV9MATH, PV10MATH
  FROM ML.PREDICT(
    MODEL `pisa.m_profil_2025`,
    (SELECT {PROFIL}, W_FSTUWT,
       PV1MATH, PV2MATH, PV3MATH, PV4MATH, PV5MATH,
       PV6MATH, PV7MATH, PV8MATH, PV9MATH, PV10MATH
     FROM `pisa.ogrenci`
     WHERE CNT = 'TUR' AND CYCLE = 2025))
),
uzun AS (
  SELECT * FROM atama
  UNPIVOT (puan FOR pv_no IN (
    PV1MATH, PV2MATH, PV3MATH, PV4MATH, PV5MATH,
    PV6MATH, PV7MATH, PV8MATH, PV9MATH, PV10MATH))
),
pv_ort AS (
  SELECT profil, pv_no, SUM(w * puan) / SUM(w) AS ort, SUM(w) AS agirlik
  FROM uzun
  GROUP BY profil, pv_no
)
SELECT
  profil,
  ROUND(AVG(ort), 1) AS matematik,
  ROUND(100 * AVG(agirlik) / SUM(AVG(agirlik)) OVER (), 1) AS ogrenci_yuzdesi
FROM pv_ort
GROUP BY profil
ORDER BY matematik DESC
""")

✔ 44.4 MB tarandı · 0.9 sn · önbellek: hayır


,profil,matematik,ogrenci_yuzdesi
0,1,509.2,22.4
1,3,461.8,8.2
2,4,461.0,39.8
3,2,426.1,29.5


---
## Slayt 33 · Senaryo simülasyonu
`sql/14_senaryo_simulasyonu.sql`

> 14 · Geleceğe dönük planlama 1: "Ne olurdu?" senaryo simülasyonu
> Slayt: "Planlama 1: senaryo simülasyonu"
> GEREKSİNİM: 06 dosyasındaki pisa.m_risk_2025 modeli.
> Ne yapar: Öğrencilerin özelliklerini bir senaryoya göre değiştirir, modelle
>   Düzey 2 altında kalma olasılığını yeniden tahmin eder ve nüfusa
>   (ağırlıklarla) genelleştirir.
> UYARI: Bu bir NEDENSEL etki tahmini değildir. Model ilişkileri öğrenir;
>   senaryolar "bu ilişkiler sabit kalırsa" sorusunu yanıtlar. Politika
>   kararı için deneysel/yarı deneysel kanıtla birlikte kullanın.

Senaryolar, tabloda bulunan değişkenlere göre otomatik kurulur. Üretilen SQL'i görmek için: `print(doldur(senaryo_sql()))`

In [ ]:
calistir(senaryo_sql())

✔ 43.1 MB tarandı · 0.8 sn · önbellek: hayır


,senaryo,duzey2_alti_tahmini_yuzde,mevcuda_gore_puan
0,0 Mevcut durum,38.5,0.0
1,"1 Okul aidiyeti +0,5 SS",37.6,-0.8
2,4 Sınıf tekrarı olmasaydı,34.4,-4.0
3,"5 Dezavantajlılara destek (ESCS < −1 için +0,5)",36.5,-1.9


---
## Slayt 34 · Okul düzeyinde erken uyarı
`sql/15_okul_erken_uyari.sql`

> 15 · Geleceğe dönük planlama 2: okul düzeyinde erken uyarı ve kaynak hedefleme
> Slayt: "Planlama 2: erken uyarı ve 2029"
> GEREKSİNİM: 06 dosyasındaki pisa.m_risk_2025 modeli.
> PISA'da okul kimlikleri anonimdir: bu sorgu YÖNTEMİ gösterir. Aynı akış
> ulusal verilerde (ör. ABİDE, LGS, e-Okul) gerçek okul listesi üretir.
> Not: Model bu öğrencilerin %80'iyle eğitildi; okul sıralaması örneklem içidir.

**A) Tahmini riski en yüksek 15 okul**

In [ ]:
calistir(r"""
WITH tahmin AS (
  SELECT CNTSCHID, W_FSTUWT AS w,
    (SELECT prob FROM UNNEST(predicted_duzey2_alti_probs) WHERE label = 1) AS risk
  FROM ML.PREDICT(MODEL `pisa.m_risk_2025`,
    (SELECT CNTSCHID, W_FSTUWT, {OZELLIKLER}
     FROM `pisa.ogrenci`
     WHERE CNT = 'TUR' AND CYCLE = 2025))
),
okul AS (
  SELECT CNTSCHID, COUNT(*) AS ogrenci,
    SUM(w * risk) / SUM(w) AS risk,
    SUM(w * risk) AS riskli_ogrenci_tahmini
  FROM tahmin GROUP BY CNTSCHID
),
sirali AS (
  SELECT *,
    ROW_NUMBER() OVER (ORDER BY risk DESC) AS sira,
    COUNT(*) OVER () AS toplam_okul,
    SUM(riskli_ogrenci_tahmini) OVER (ORDER BY risk DESC
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
      / SUM(riskli_ogrenci_tahmini) OVER () AS kumulatif_pay
  FROM okul
)
SELECT sira, CNTSCHID AS okul_kimligi, ogrenci,
  ROUND(100 * risk, 1) AS tahmini_risk_yuzdesi
FROM sirali
ORDER BY sira
LIMIT 15
""")

✔ 24.8 MB tarandı · 0.6 sn · önbellek: hayır


,sira,okul_kimligi,ogrenci,tahmini_risk_yuzdesi
0,1,79200002,1,72.6
1,2,79200196,40,63.3
2,3,79200182,40,60.5
3,4,79200163,40,58.5
4,5,79200137,39,58.4
5,6,79200145,17,57.3
6,7,79200087,41,56.4
7,8,79200203,21,54.7
8,9,79200159,42,54.7
9,10,79200055,41,54.6


**B) Yoğunlaşma: risk altındaki öğrencilerin yarısı okulların yüzde kaçında?**

In [ ]:
calistir(r"""
WITH tahmin AS (
  SELECT CNTSCHID, W_FSTUWT AS w,
    (SELECT prob FROM UNNEST(predicted_duzey2_alti_probs) WHERE label = 1) AS risk
  FROM ML.PREDICT(MODEL `pisa.m_risk_2025`,
    (SELECT CNTSCHID, W_FSTUWT, {OZELLIKLER}
     FROM `pisa.ogrenci`
     WHERE CNT = 'TUR' AND CYCLE = 2025))
),
okul AS (
  SELECT CNTSCHID, COUNT(*) AS ogrenci,
    SUM(w * risk) / SUM(w) AS risk,
    SUM(w * risk) AS riskli_ogrenci_tahmini
  FROM tahmin GROUP BY CNTSCHID
),
sirali AS (
  SELECT *,
    ROW_NUMBER() OVER (ORDER BY risk DESC) AS sira,
    COUNT(*) OVER () AS toplam_okul,
    SUM(riskli_ogrenci_tahmini) OVER (ORDER BY risk DESC
      ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
      / SUM(riskli_ogrenci_tahmini) OVER () AS kumulatif_pay
  FROM okul
)
SELECT
  MIN(sira) AS okul_sayisi,
  ANY_VALUE(toplam_okul) AS toplam_okul,
  ROUND(100 * MIN(sira) / ANY_VALUE(toplam_okul), 1) AS okullarin_yuzdesi
FROM sirali
WHERE kumulatif_pay >= 0.5
""")

✔ 24.8 MB tarandı · 1.0 sn · önbellek: hayır


,okul_sayisi,toplam_okul,okullarin_yuzdesi
0,82,203,40.4


---
## Slayt 35 · PISA 2029 senaryoları
`sql/16_trend_projeksiyon_2029.sql`

> 16 · Geleceğe dönük planlama 3: Türkiye için 2029 senaryoları
> Slayt: "Planlama 2: erken uyarı ve 2029"
> Veri: MEB/OECD raporlarındaki Türkiye ortalamaları (2003–2025, 8 döngü).
> Neden ARIMA / derin öğrenme değil? 8 noktalık, düzensiz aralıklı bir seri
> karmaşık bir zaman serisi modeli için çok kısadır. Burada şeffaf bir doğrusal
> eğilim ve iki basit senaryo kullanılır; ±2 artık standart sapması
> belirsizliğin büyüklüğünü gösterir. Sonuç bir "hedef aralığı"dır, kehanet değildir.

In [ ]:
calistir(r"""
WITH rapor AS (
  SELECT * FROM UNNEST([
    STRUCT(2003 AS yil, 423 AS matematik, 441 AS okuma, 434 AS fen),
    STRUCT(2006, 424, 447, 424),
    STRUCT(2009, 445, 464, 454),
    STRUCT(2012, 448, 475, 463),
    STRUCT(2015, 420, 428, 425),
    STRUCT(2018, 454, 466, 468),
    STRUCT(2022, 453, 456, 476),
    STRUCT(2025, 462, 472, 494)
  ])
),
uzun AS (
  SELECT yil, alan, puan
  FROM rapor
  UNPIVOT (puan FOR alan IN (matematik, okuma, fen))
),
egilim AS (
  SELECT alan,
    COVAR_POP(yil, puan) / VAR_POP(yil) AS egim,
    AVG(puan) - COVAR_POP(yil, puan) / VAR_POP(yil) * AVG(yil) AS sabit,
    MAX(IF(yil = 2025, puan, NULL)) AS p2025,
    MAX(IF(yil = 2022, puan, NULL)) AS p2022
  FROM uzun GROUP BY alan
),
artik AS (
  SELECT alan, SQRT(SUM(POW(puan - (sabit + egim * yil), 2)) / (COUNT(*) - 2)) AS sd
  FROM uzun JOIN egilim USING (alan)
  GROUP BY alan
)
SELECT
  alan,
  ROUND(egim, 2)                          AS yillik_egim,
  p2025                                   AS puan_2025,
  p2025                                   AS s1_duzey_korunursa,
  ROUND(sabit + egim * 2029)              AS s2_uzun_donem_egilim,
  ROUND(2 * sd)                           AS s2_belirsizlik_arti_eksi,
  ROUND(p2025 + (p2025 - p2022) / 3 * 4)  AS s3_son_donem_hizi_surerse
FROM egilim JOIN artik USING (alan)
ORDER BY alan
""")

✔ 0.0 MB tarandı · 0.3 sn · önbellek: hayır


,alan,yillik_egim,puan_2025,s1_duzey_korunursa,s2_uzun_donem_egilim,s2_belirsizlik_arti_eksi,s3_son_donem_hizi_surerse
0,fen,2.61,494,494,495.0,33.0,518.0
1,matematik,1.58,462,462,465.0,24.0,474.0
2,okuma,0.84,472,472,469.0,32.0,493.0


---
## Slayt 36 · Üretken YZ: açık uçlu kodlama (isteğe bağlı)
`sql/09_ai_classify_istege_bagli.sql`

> 09 · (İSTEĞE BAĞLI) Üretken YZ ile açık uçlu yanıt kodlama
> Slayt: "BigQuery'de SQL içinden yapay zekâ"
> GEREKSİNİM: Faturalandırması açık bir proje (sandbox'ta çalışmaz),
>             BigQuery'nin yönetilen YZ fonksiyonlarına erişim. Gemini
>             çağrıları ayrıca ücretlendirilir; az satırla deneyin.
> Parametre adlarını güncel dokümandan teyit edin:
>   cloud.google.com/bigquery/docs/reference/standard-sql/bigqueryml-syntax-ai-classify
> AŞAĞIDAKİ YANITLAR DEMO İÇİN UYDURULMUŞ ÖRNEKLERDİR; gerçek öğrenci verisi değildir.
> Kamuya açık PISA dosyalarında açık uçlu yanıt metinleri bulunmaz.

**A) YZ ile kodla ve sonucu tabloya yaz (YZ bir kez çağrılır)**

In [ ]:
calistir(r"""
CREATE OR REPLACE TABLE `pisa.yz_kodlama` AS
WITH yanitlar AS (
  SELECT * FROM UNNEST([
    STRUCT(1 AS ogrenci, 'Bitkiler güneş ışığıyla fotosentez yapıp kendi besinlerini üretir.' AS yanit, 'tam doğru' AS uzman_kodu),
    STRUCT(2, 'Güneş ışığından energy alıp karbondioksit ve sudan şeker yaparlar.', 'tam doğru'),
    STRUCT(3, 'Işık olmazsa bitki ölür.', 'kısmen doğru'),
    STRUCT(4, 'Büyümek için ışık lazım, yeşil kalmak için.', 'kısmen doğru'),
    STRUCT(5, 'Bitkiler ışığı topraktan alır.', 'yanlış'),
    STRUCT(6, 'Isınmak için güneşe ihtiyaç duyarlar, yoksa donarlar.', 'yanlış'),
    STRUCT(7, 'bilmiyorum', 'boş veya ilgisiz'),
    STRUCT(8, 'Klorofil ışığı soğurur ve bu enerjiyle glikoz üretilir, oksijen açığa çıkar.', 'tam doğru'),
    STRUCT(9, 'Güneş bitkilere vitamin verir.', 'yanlış'),
    STRUCT(10, 'Fotosentez için.', 'kısmen doğru')
  ])
)
SELECT
  ogrenci, yanit, uzman_kodu,
  AI.CLASSIFY(
    CONCAT('Soru: Bitkiler neden güneş ışığına ihtiyaç duyar? Öğrenci yanıtı: ', yanit),
    categories => ['tam doğru', 'kısmen doğru', 'yanlış', 'boş veya ilgisiz']
  ) AS yz_kodu
FROM yanitlar
""")

BadRequest: 400 GET https://bigquery.googleapis.com/bigquery/v2/projects/atolye-0926/queries/597dc487-6fad-48ae-8a6b-7ca50ffbc75a?maxResults=0&location=EU&prettyPrint=false: Cloud AI API is not enabled: Agent Platform API has not been used in project 576566327398 before or it is disabled. Enable it by visiting https://console.developers.google.com/apis/api/aiplatform.googleapis.com/overview?project=576566327398 then retry. If you enabled this API recently, wait a few minutes for the action to propagate to our systems and retry.

Location: EU
Job ID: 597dc487-6fad-48ae-8a6b-7ca50ffbc75a


In [ ]:
from googleapiclient import discovery
from google.colab import auth

try:
    # Ensure we use the active credentials
    auth.authenticate_user()

    # Build the service usage API client
    service = discovery.build('serviceusage', 'v1')

    # Check the status of Vertex AI API (aiplatform.googleapis.com)
    name = f'projects/{PROJE_ID}/services/aiplatform.googleapis.com'
    request = service.services().get(name=name)
    response = request.execute()

    state = response.get('state', 'UNKNOWN')
    if state == 'ENABLED':
        print(f"✔ Vertex AI API ({PROJE_ID}) projenizde ETKİN durumdadır.")
    else:
        print(f"❌ Vertex AI API ({PROJE_ID}) projenizde ETKİN DEĞİLDİR (Durum: {state}).")
except Exception as e:
    print(f"API durumu kontrol edilirken bir hata oluştu: {e}")

❌ Vertex AI API (atolye-0926) projenizde ETKİN DEĞİLDİR (Durum: DISABLED).


**B) Sonuçlar yan yana**

In [ ]:
calistir(r"""
SELECT ogrenci, yanit, uzman_kodu, yz_kodu, uzman_kodu = yz_kodu AS uyumlu
FROM `pisa.yz_kodlama`
ORDER BY ogrenci
""")

NotFound: 404 Not found: Table atolye-0926:pisa.yz_kodlama was not found in location EU; reason: notFound, message: Not found: Table atolye-0926:pisa.yz_kodlama was not found in location EU

Location: EU
Job ID: 25fc3d3d-c387-48dd-99d9-4f24bcdd7168


**C) Uzman ile YZ arasında uyum: yüzde uyum ve Cohen kappa**

In [ ]:
calistir(r"""
WITH n AS (SELECT COUNT(*) AS toplam FROM `pisa.yz_kodlama`),
po AS (SELECT AVG(IF(uzman_kodu = yz_kodu, 1, 0)) AS p FROM `pisa.yz_kodlama`),
pe AS (
  SELECT SUM(u.adet * y.adet) / POW(ANY_VALUE(n.toplam), 2) AS p
  FROM (SELECT uzman_kodu AS kod, COUNT(*) AS adet FROM `pisa.yz_kodlama` GROUP BY kod) AS u
  JOIN (SELECT yz_kodu AS kod, COUNT(*) AS adet FROM `pisa.yz_kodlama` GROUP BY kod) AS y USING (kod)
  CROSS JOIN n
)
SELECT ROUND(po.p, 2) AS yuzde_uyum, ROUND((po.p - pe.p) / (1 - pe.p), 2) AS cohen_kappa
FROM po CROSS JOIN pe
""")

---
<sub>© 2026 Zekeriya Beşiroğlu · İstanbul Data Science Academy · CMEEP 2026 Çalıştay 14</sub>